# Load Dataset

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import importlib.util
import json
import os
import sys
from collections import Counter
from pathlib import Path

# ── Third-party ───────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
from lightgbm import Booster, LGBMClassifier
from matplotlib.colors import LogNorm
from sklearn import multiclass
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

# ── Project source modules ─────────────────────────────────────────────────────
PROJECT_ROOT = Path("..").resolve()
SRC_DIR = PROJECT_ROOT / "src"

def _load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

sleep_data_utils = _load_module("sleep_data_utils", SRC_DIR / "sleep_data_utils.py")
sleep_model_utils = _load_module("sleep_model_utils", SRC_DIR / "sleep_model_utils.py")


DEFAULT_STAGE_MAP = sleep_data_utils.DEFAULT_STAGE_MAP
SleepDataset = sleep_data_utils.SleepDataset
build_subject_list = sleep_data_utils.build_subject_list
encode_labels = sleep_data_utils.encode_labels
flatten_dataset = sleep_data_utils.flatten_dataset
load_sleep_config = sleep_data_utils.load_sleep_config
load_or_create_subject_split = sleep_data_utils.load_or_create_subject_split
render_subject_split_file_template = sleep_data_utils.render_subject_split_file_template

plot_confusion_matrix = sleep_model_utils.plot_confusion_matrix


# ── Config ─────────────────────────────────────────────────────────────────────
CONFIG_PATH = PROJECT_ROOT / "config" / "sleep-model.yaml"
cfg = load_sleep_config(CONFIG_PATH)

SMALL_BATCH_TEST = cfg["subjects"]["small_batch_test"]
FEAT_COLS = cfg["data"]["feature_cols"]
TARGET_COL = cfg["data"]["target_cols"]
FEAT_METADATA_COLS = cfg["data"]["feature_metadata_cols"]
STAGE_MAP = cfg.get("labels", {}).get("stage_map", DEFAULT_STAGE_MAP)

SIGNAL_DIR = str((CONFIG_PATH.parent / cfg["data"]["signal_dir"]).resolve())
METADATA_CSV = str((CONFIG_PATH.parent / cfg["data"]["metadata_csv"]).resolve())

ACROSS_SUBJECT_RATIO = cfg["split"]["across_subject_ratio"]
subject_split_template = cfg["split"].get(
    "subject_split_file",
    "../data/processed/splits/subject_split_seed{seed}_ratio_{ratio_tag}.json",
)
SPLIT_FILE = str((
    CONFIG_PATH.parent / render_subject_split_file_template(
        subject_split_template,
        seed=cfg["split"]["seed"],
        ratio=ACROSS_SUBJECT_RATIO,
    )
).resolve())

SUBJECTS = build_subject_list(
    prefix=cfg["subjects"]["prefix"],
    start=cfg["subjects"]["start"],
    end=cfg["subjects"]["end"],
    small_batch_test=SMALL_BATCH_TEST,
    small_batch_limit=cfg["subjects"]["small_batch_limit"],
)

In [ ]:
# Build/load subject split using config seed + ratio
split_payload = load_or_create_subject_split(
    subject_ids=SUBJECTS,
    split_file_path=SPLIT_FILE,
    ratio=cfg["split"].get("across_subject_ratio", cfg["split"]["within_subject_ratio"]),
    seed=cfg["split"]["seed"],
)

train_subs = split_payload["train_subjects"]
val_subs = split_payload["val_subjects"]
test_subs = split_payload["test_subjects"]

print(f"Loaded split file: {SPLIT_FILE}")
print(f"Subjects => train: {len(train_subs)}, val: {len(val_subs)}, test: {len(test_subs)}")

train_ds = SleepDataset(
    signals_dir=SIGNAL_DIR,
    metadata_csv=METADATA_CSV,
    feature_cols=FEAT_COLS,
    label_cols=TARGET_COL,
    allowed_subjects=train_subs,
    block_duration_sec=cfg["dataset"]["block_duration_sec"],
    epoch_sec=cfg["dataset"]["epoch_sec"],
    drop_boundary=cfg["dataset"]["drop_boundary"],
    meta_feature_cols=FEAT_METADATA_COLS,
)

val_ds = SleepDataset(
    signals_dir=SIGNAL_DIR,
    metadata_csv=METADATA_CSV,
    feature_cols=FEAT_COLS,
    label_cols=TARGET_COL,
    allowed_subjects=val_subs,
    block_duration_sec=cfg["dataset"]["block_duration_sec"],
    epoch_sec=cfg["dataset"]["epoch_sec"],
    drop_boundary=cfg["dataset"]["drop_boundary"],
    meta_feature_cols=FEAT_METADATA_COLS,
)

test_ds = SleepDataset(
    signals_dir=SIGNAL_DIR,
    metadata_csv=METADATA_CSV,
    feature_cols=FEAT_COLS,
    label_cols=TARGET_COL,
    allowed_subjects=test_subs,
    block_duration_sec=cfg["dataset"]["block_duration_sec"],
    epoch_sec=cfg["dataset"]["epoch_sec"],
    drop_boundary=cfg["dataset"]["drop_boundary"],
    meta_feature_cols=FEAT_METADATA_COLS,
)

X_train, y_train, info_train = flatten_dataset(train_ds)
X_val, y_val, info_val = flatten_dataset(val_ds)
X_test, y_test, info_test = flatten_dataset(test_ds)

X_train, y_train, info_train = encode_labels(X_train, y_train, info_train, stage_map=STAGE_MAP)
X_val, y_val, info_val = encode_labels(X_val, y_val, info_val, stage_map=STAGE_MAP)
X_test, y_test, info_test = encode_labels(X_test, y_test, info_test, stage_map=STAGE_MAP)

print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_val shape:   {X_val.shape} | y_val shape:   {y_val.shape}")
print(f"X_test shape:  {X_test.shape} | y_test shape:  {y_test.shape}")

In [ ]:
print(f"Signal features:   {len(FEAT_COLS)}")
print(f"Metadata features: {len(FEAT_METADATA_COLS)}")
print(f"Total input dim:   {len(FEAT_COLS) + len(FEAT_METADATA_COLS)}")

print("\nClass distribution (train):")
values, counts = np.unique(y_train, return_counts=True)
labels = [k for k, _ in sorted(STAGE_MAP.items(), key=lambda kv: kv[1])]
for v, c in zip(values, counts):
    print(f"  {labels[v]}: {c} ({c / counts.sum():.2%})")

# Feature Engineering

## add power ratios / lag features

In [ ]:
def add_core_sleep_ratios_df(X, ratio_cols, original_cols=None, eps=1e-6):
    """
    Add relative band-power and ratio features for each EEG channel.

    Parameters
    ----------
    X : np.ndarray, shape (n_samples, n_features)
    ratio_cols : list[str]   EEG channel prefixes, e.g. ["eeg_c4", "eeg_f4"]
    original_cols : list[str] or None
        Column names for X.  Defaults to FEAT_COLS + FEAT_METADATA_COLS.
    """
    if original_cols is None:
        original_cols = FEAT_COLS + FEAT_METADATA_COLS

    df = pd.DataFrame(X, columns=original_cols)

    for ch in ratio_cols:
        delta = df[f"{ch}_bp_delta"]
        theta = df[f"{ch}_bp_theta"]
        alpha = df[f"{ch}_bp_alpha"]
        sigma = df[f"{ch}_bp_sigma"]
        beta  = df[f"{ch}_bp_beta"]

        total = delta + theta + alpha + sigma + beta

        df[f"{ch}_bp_delta_rel"] = delta / (total + eps)
        df[f"{ch}_bp_theta_rel"] = theta / (total + eps)
        df[f"{ch}_bp_alpha_rel"] = alpha / (total + eps)
        df[f"{ch}_bp_sigma_rel"] = sigma / (total + eps)
        df[f"{ch}_bp_beta_rel"]  = beta  / (total + eps)

        df[f"{ch}_bp_delta_over_theta"] = delta / (theta + eps)
        df[f"{ch}_bp_delta_over_alpha"] = delta / (alpha + eps)
        df[f"{ch}_bp_theta_over_alpha"] = theta / (alpha + eps)
        df[f"{ch}_bp_sigma_over_delta"] = sigma / (delta + eps)

    return df.to_numpy(dtype=np.float32), df.columns.tolist()

In [ ]:
def add_lag_features(X, feature_names, lag_features, lags=(1, 2, 3)):
    """
    Add lag features to selected columns.
    """
    df = pd.DataFrame(X, columns=feature_names)

    for col in lag_features:
        if col not in df.columns:
            raise ValueError(f"{col} not found in feature_names")
        for lag in lags:
            df[f"{col}_lag{lag}"] = df[col].shift(lag)

    keep_idx = np.arange(max(lags), len(df))
    df = df.iloc[keep_idx].reset_index(drop=True)

    feature_names_new = df.columns.tolist()
    X_new = df.to_numpy(dtype=np.float32)

    return X_new, feature_names_new, keep_idx

In [ ]:
eeg_channels = ["eeg_c4", "eeg_f4", "eeg_o2", "eeg_fp1", "eeg_t3", "eeg_cz"]

lag_features = [
    "eeg_c4_bp_delta",
    "eeg_o2_bp_theta",
    "eog_e2_bp_slow",
    "eeg_cz_bp_delta",
    "eeg_cz_bp_high_gamma",
    "hr_mean",
    "temp_mean",
]
lag_shifts = (1,2,3)


X_train_aug, feature_names_aug0 = add_core_sleep_ratios_df(X_train, ratio_cols=eeg_channels)
X_val_aug, _ = add_core_sleep_ratios_df(X_val, ratio_cols=eeg_channels)
X_test_aug, _ = add_core_sleep_ratios_df(X_test, ratio_cols=eeg_channels)


print("Original train:", X_train.shape)
print("Added power ratio train:", X_train_aug.shape)

X_train_aug, feature_names_aug, keep_idx = add_lag_features(
    X_train_aug,
    feature_names=feature_names_aug0,
    lag_features=lag_features,
    lags=lag_shifts,
)
y_train_aug = y_train[keep_idx]
info_train_aug = info_train.iloc[keep_idx].reset_index(drop=True)

X_val_aug, _, keep_idx = add_lag_features(
    X_val_aug,
    feature_names=feature_names_aug0,
    lag_features=lag_features,
    lags=lag_shifts,
)
y_val_aug = y_val[keep_idx]
info_val_aug = info_val.iloc[keep_idx].reset_index(drop=True)

X_test_aug, _, keep_idx = add_lag_features(
    X_test_aug,
    feature_names=feature_names_aug0,
    lag_features=lag_features,
    lags=lag_shifts,
)
y_test_aug = y_test[keep_idx]
info_test_aug = info_test.iloc[keep_idx].reset_index(drop=True)

print(f"Added {len(lag_shifts)} lags :", X_train_aug.shape)
print("new features:", feature_names_aug)

In [ ]:
# override
X_train = X_train_aug
y_train = y_train_aug
info_train = info_train_aug
X_val = X_val_aug
y_val = y_val_aug
info_val = info_val_aug
X_test = X_test_aug
y_test = y_test_aug
info_test = info_test_aug

# Model train and eval

## Logistic Regression

In [ ]:
c_list = [0.01, 1.0, 100.0]

for c in c_list:
    logreg = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
                    max_iter=2000,
                    solver="lbfgs",
                    class_weight="balanced",
                    C=c,
                    random_state=42,
                ))
    ])

    logreg.fit(X_train, y_train)
    val_pred = logreg.predict(X_val)

    print("Val macro F1:", f1_score(y_val, val_pred, average="macro"), " for C=", c)
    print(classification_report(y_val, val_pred, digits=4, target_names=["W","N1","N2","N3","R"]))
    print("===================")

As C increases, regularizations are weaker. Increased score for weaker means that model can benifit from having many weaker signals. It also means that logistic regression tends to underfit for the given data. 

## XGBoost

In [ ]:
def objective_xgb(trial, X, y, groups, n_splits=4, class_weight=None):
    params = {
        "objective": "multi:softprob",
        "num_class": 5,
        "eval_metric": "mlogloss",
        "tree_method": "hist",
        "random_state": 42,
        "n_jobs": -1,

        "n_estimators": trial.suggest_int("n_estimators", 300, 900),
        "max_depth": trial.suggest_int("max_depth", 3, 7),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.12, log=True),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 6),
        "gamma": trial.suggest_float("gamma", 0.0, 2.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 5.0, log=True),
    }

    cv = GroupKFold(n_splits=n_splits)
    fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y, groups), start=1):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        w_tr = compute_sample_weight(class_weight="balanced", y=y_tr)

        model = XGBClassifier(**params)
        model.fit(X_tr, y_tr, sample_weight=w_tr)

        y_pred = model.predict(X_val)
        score = f1_score(y_val, y_pred, average="macro")
        fold_scores.append(score)

        trial.report(np.mean(fold_scores), step=fold)
        print(f"[Trial {trial.number}] Fold {fold} | F1: {score:.4f} | Mean: {np.mean(fold_scores):.4f}")

        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(fold_scores))

In [ ]:
groups = info_train["subject_id"].values

optuna.logging.set_verbosity(optuna.logging.INFO)

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=2),
)

study.optimize(
    lambda trial: objective_xgb(trial, X_train, y_train, groups),
    n_trials=30,
    show_progress_bar=True,
)

print("Best trial score:", study.best_value)
print("Best params:")
print(study.best_params)

In [ ]:
best_params = study.best_params.copy()
best_xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=5,
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
    **best_params,
)

best_xgb.fit(X_train, y_train)

y_pred = best_xgb.predict(X_val)
print("Val macro F1:", f1_score(y_val, y_pred, average="macro"))
print(classification_report(
    y_val, y_pred, digits=4,
    target_names=["W", "N1", "N2", "N3", "R"]
))
val_pred=y_pred

In [ ]:
# save model
exp_dir = Path("../models/xgb_featured")
exp_dir.mkdir(parents=True, exist_ok=True)

best_xgb.save_model(exp_dir / "model.json")

artifact = {
    "best_params": study.best_params,
    "best_cv_score": float(study.best_value),
    "feature_cols": feature_names_aug,
}
with open(exp_dir / "metadata.json", "w") as f:
    json.dump(artifact, f, indent=2)

print(f"Saved XGBoost to {exp_dir}")

In [ ]:
importance = best_xgb.feature_importances_
feat_imp = pd.Series(importance, index=feature_names_aug).sort_values(ascending=False)
feat_imp.head(20)

In [ ]:
val_df = info_val.copy()
val_df["y_true"] = y_val
val_df["y_pred"] = val_pred
scores = []
for sid, g in val_df.groupby("subject_id"):
    scores.append(f1_score(g.y_true, g.y_pred, average="macro"))
plt.hist(scores, bins=30, alpha=0.7)
plt.xlabel("f1 scores per subject")
plt.ylabel("Frequency")
plt.title("Distribution of macro F1 per subject")
plt.show()

In [ ]:
cm = {"confusion_matrix":confusion_matrix(y_val,y_pred_xgb)}
plot_confusion_matrix(cm, STAGE_MAP, title="XGBoost validation confusion")

## LightGBM

In [ ]:
def objective_lgb(trial, X, y, groups, n_splits=4, class_weight=None):
    params = {
        "objective": "multiclass",
        "num_class": 5,
        "metric": "multi_logloss",
        "random_state": 42,
        "n_jobs": -1,
        "verbosity": -1,

        "n_estimators": trial.suggest_int("n_estimators", 300, 900),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.12, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 16, 96),
        "max_depth": trial.suggest_categorical("max_depth", [-1, 3, 4, 5, 6, 8]),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 60),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 5.0, log=True),
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 1.0),
    }

    cv = GroupKFold(n_splits=n_splits)
    fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y, groups), start=1):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        w_tr = w_tr = compute_sample_weight(class_weight="balanced", y=y_tr)

        model = LGBMClassifier(**params)
        model.fit(X_tr, y_tr, sample_weight=w_tr)

        y_pred = model.predict(X_val)
        score = f1_score(y_val, y_pred, average="macro")
        fold_scores.append(score)

        trial.report(np.mean(fold_scores), step=fold)
        print(f"[Trial {trial.number}] Fold {fold} | F1: {score:.4f} | Mean: {np.mean(fold_scores):.4f}")

        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(fold_scores))

In [ ]:
groups = info_train["subject_id"].values

optuna.logging.set_verbosity(optuna.logging.INFO)

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=2),
)

study.optimize(
    lambda trial: objective_lgb(trial, X_train, y_train, groups),
    n_trials=50,
    show_progress_bar=True,
)

print("Best trial score:", study.best_value)
print("Best params:")
print(study.best_params)

In [ ]:
best_params = study.best_params.copy()

best_lgb = LGBMClassifier(
    objective="multiclass",
    num_class=5,
    metric="multi_logloss",
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
    **best_params,
)

best_lgb.fit(X_train, y_train)

y_pred = best_lgb.predict(X_val)

print("Val macro F1:", f1_score(y_val, y_pred, average="macro"))
print(classification_report(
    y_val, y_pred, digits=4,
    target_names=["W", "N1", "N2", "N3", "R"]
))

In [ ]:
# save model
exp_dir = Path("../models/lgb_featured")
exp_dir.mkdir(parents=True, exist_ok=True)

best_lgb.booster_.save_model(str(exp_dir / "model.txt"))

artifact = {
    "best_params": best_params,
    "best_cv_score": float(study.best_value),
    "feature_cols": feature_names_aug,
}
with open(exp_dir / "metadata.json", "w") as f:
    json.dump(artifact, f, indent=2)

print(f"Saved LightGBM to {exp_dir}")

In [ ]:
imp_gain = pd.Series(
    best_lgb.booster_.feature_importance(importance_type="gain"),
    index=feature_names_aug
).sort_values(ascending=False)

print(imp_gain.head(20))

In [ ]:
val_df = info_val.copy()
val_df["y_true"] = y_val
val_df["y_pred"] = y_pred
scores = []
for sid, g in val_df.groupby("subject_id"):
    scores.append(f1_score(g.y_true, g.y_pred, average="macro"))
plt.hist(scores, bins=30, alpha=0.7)
plt.xlabel("f1 scores per subject")
plt.ylabel("Frequency")
plt.title("Distribution of macro F1 per subject")
plt.show()

In [ ]:
cm = {"confusion_matrix":confusion_matrix(y_val,y_pred_lgb)}
plot_confusion_matrix(cm, STAGE_MAP, title="LightGBM validation confusion")

The reason why N1 recall is low is because it is often mistaken as N2 (the dominant class) and W.

## post process: smoothing
mode filter for smoothing

In [ ]:
best_xgb = globals().get("best_xgb", None)
best_lgb = globals().get("best_lgb", None)

if best_xgb is None:
    xgb_model_path = Path("../models/xgb_featured/model.json")
    if not xgb_model_path.exists():
        raise FileNotFoundError(f"XGBoost model not found: {xgb_model_path}")

    best_xgb = XGBClassifier()
    best_xgb.load_model(xgb_model_path)
    print(f"Loaded XGBoost from: {xgb_model_path}")

if best_lgb is None:
    lgb_model_path = Path("../models/lgb_featured/model.txt")
    if not lgb_model_path.exists():
        raise FileNotFoundError(f"LightGBM model not found: {lgb_model_path}")

    class _LGBBoosterWrapper:
        def __init__(self, booster):
            self.booster = booster

        def predict_proba(self, X):
            return self.booster.predict(X)

        def predict(self, X):
            return np.argmax(self.predict_proba(X), axis=1)

    best_lgb = _LGBBoosterWrapper(Booster(model_file=str(lgb_model_path)))
    print(f"Loaded LightGBM from: {lgb_model_path}")

In [ ]:
def mode_filter_1d(y, kernel_size=5):
    assert kernel_size % 2 == 1, "kernel_size must be odd"
    pad = kernel_size // 2
    y = np.asarray(y)
    y_pad = np.pad(y, (pad, pad), mode="edge")
    out = np.empty_like(y)

    for i in range(len(y)):
        window = y_pad[i:i + kernel_size]
        cnt = Counter(window)
        max_count = max(cnt.values())

        # tie-break: keep center label if tied
        candidates = [k for k, v in cnt.items() if v == max_count]
        center = y[i]
        out[i] = center if center in candidates else candidates[0]

    return out

In [ ]:
y_pred_xgb = best_xgb.predict(X_val)
y_pred_lgb = best_lgb.predict(X_val)

y_pred_xgb_smooth = mode_filter_1d(y_pred_xgb, kernel_size=3)
print("XGBoost Val macro F1:", f1_score(y_val, y_pred_xgb_smooth, average="macro"))
print(classification_report(y_val, y_pred_xgb_smooth, digits=4, target_names=["W","N1","N2","N3","R"]))

y_pred_lgb_smooth = mode_filter_1d(y_pred_lgb, kernel_size=3)
print("LightGBM Val macro F1:", f1_score(y_val, y_pred_lgb_smooth, average="macro"))
print(classification_report(y_val, y_pred_lgb_smooth, digits=4, target_names=["W","N1","N2","N3","R"]))

## combine xgb and lgb

In [ ]:
y_pred_xgb = best_xgb.predict_proba(X_val)
y_pred_lgb = best_lgb.predict_proba(X_val)

y_pred_comb = np.argmax(y_pred_xgb * 0.5 + y_pred_lgb * 0.5, axis=1)
print("Val macro F1:", f1_score(y_val, y_pred_comb, average="macro"))
print(classification_report(
    y_val, y_pred_comb, digits=4,
    target_names=["W", "N1", "N2", "N3", "R"]
))